# refined_layer

In [0]:
%sql
-- DROP TABLE IF EXISTS workspace.refined_upi.transactions_silver;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
# Read tables 
df_txn = spark.read.table("workspace.raw_upi.transactions_bronze")
df_merchants = spark.read.table("workspace.raw_upi.merchants_bronze")
df_fraud = spark.read.table("workspace.raw_upi.fraud_flags_bronze")
# latest record per transaction
window_spec = Window.partitionBy("transaction_id").orderBy(
    col("_ingest_timestamp").desc(),
    col("_file_name").desc()
)

df_txn = df_txn.withColumn("rn", row_number().over(window_spec)) \
               .filter(col("rn") == 1) \
               .drop("rn")

# timestamp to ist

df_txn = df_txn.withColumn(
    "transaction_timestamp_ist",
    from_utc_timestamp(col("transaction_timestamp"), "Asia/Kolkata")
)
# VPA validation

df_txn = df_txn.withColumn(
    "is_valid_vpa",
    col("sender_vpa").rlike("^[^@\\s]+@[^@\\s]+$")
)

# Time derivations

df_txn = df_txn \
    .withColumn("transaction_hour", hour(col("transaction_timestamp_ist"))) \
    .withColumn("transaction_day_of_week", date_format(col("transaction_timestamp_ist"), "E")) \
    .withColumn("is_weekend", col("transaction_day_of_week").isin("Sat", "Sun")) \
    .withColumn(
        "amount_bucket",
        when(col("amount") <= 500, "0-500")
        .when((col("amount") > 500) & (col("amount") <= 2000), "500-2000")
        .when((col("amount") > 2000) & (col("amount") <= 10000), "2000-10000")
        .otherwise("10000+")
    )

# merchant enrichment

df_txn = df_txn.join(
    df_merchants.select(
        "merchant_id",
        col("merchant_category"),
        col("state").alias("merchant_state")
    ),
    df_txn.receiver_merchant_id == df_merchants.merchant_id,
    "left"
)

# fraud_enrichmnt

df_txn = df_txn.join(
    df_fraud.select(
        "transaction_id",
        "fraud_rule",
        "risk_score"
    ),
    "transaction_id",
    "left"
)

df_txn = df_txn.withColumn(
    "is_fraud",
    col("fraud_rule").isNotNull()
)

# handling failure reason
df_txn = df_txn.withColumn(
    "failure_reason",
    when(col("status") == "SUCCESS", lit(None)).otherwise(col("failure_reason"))
)

# drop duplicates

df_txn = df_txn.dropDuplicates(["transaction_id"])


# silver layer

df_txn.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.refined_upi.transactions_silver")



+--------------+---+
|transaction_id|cnt|
+--------------+---+
+--------------+---+

+--------------------+----+
|          _file_name| cnt|
+--------------------+----+
|/Volumes/workspac...|3000|
+--------------------+----+

